In [ ]:
%pip install "librosa==1.0.0" --quiet

# Module M4-CNN — เทรน spectrogram CNN จากศูนย์ บน ESC-50

ครึ่งหลังของโมดูล M4: ครึ่งแรก (RF บน MFCC — `day2/module4_sound_to_numbers/`)
บีบเสียงเป็นตาราง 40 ตัวเลขแล้วได้ accuracy ~0.45 — ที่นี่เรายกข้อจำกัดนั้น
โดยให้ CNN อ่าน **spectrogram เต็ม** และเรียนรู้ feature เอง บนข้อมูลเดียวกัน
เทียบกันที่ fold เดียวกัน

- **โมดูลนี้รันบน Kaggle GPU** — สร้าง notebook บน
  [kaggle.com/code](https://www.kaggle.com/code) แล้วตั้งค่า
  **Settings → Accelerator: GPU T4 ×2** (ใช้ตัวเดียวพอ) และ **Internet: On**
  ก่อนกด Run All
- เซลล์แรกติดตั้ง dependencies ตาม pin ของ `requirements-kaggle.txt`
  (torch **ไม่ต้องติดตั้ง** — Kaggle ให้มาแล้ว; ESC-50 ดาวน์โหลดในโน้ตบุ๊ก ~600 MB)
- รันทั้งไฟล์ใช้เวลาราว **20–25 นาที บน T4** (ดาวน์โหลดข้อมูล + MFCC/RF baseline
  + เทรน CNN)
- สถานะการตรวจ: โค้ดผ่าน lint และทดสอบ code path บน CPU แล้ว — **การรันจริงบน
  Kaggle T4 ต้องตรวจด้วยมือ 1 รอบก่อนวันอบรม** (ดู checklist ใน README)

**ที่มาและ license ของทุกอย่างที่โมดูลนี้ใช้**

**ที่มาและ license ของทุกอย่างที่โมดูลนี้ใช้**

| สิ่งที่ใช้ | ที่มา | license |
|---|---|---|
| ESC-50 — เสียงสิ่งแวดล้อม 2,000 คลิป (5 วินาที/คลิป, 50 class × 40 คลิป, แบ่ง 5 folds มาให้แล้ว) | [github.com/karoldvl/ESC-50](https://github.com/karoldvl/ESC-50) · [paper: Piczak, ACM MM 2015](http://karol.piczak.com/papers/Piczak2015-ESC-Dataset.pdf) | **CC BY-NC 3.0** ⚠️ **non-commercial** — ใช้ในห้องเรียนได้ แต่ **ห้ามแจกต่อเชิงพาณิชย์** (ดูหมายเหตุด้านล่าง) |
| librosa (โหลดเสียง + MFCC/spectrogram) | [github.com/librosa/librosa](https://github.com/librosa/librosa) | **ISC** |
| scikit-learn (random forest + metrics) | [github.com/scikit-learn/scikit-learn](https://github.com/scikit-learn/scikit-learn) | **BSD-3** |
| PyTorch (เฉพาะโมดูล CNN) | [github.com/pytorch/pytorch](https://github.com/pytorch/pytorch) | **BSD-3** |

> ⚠️ **หมายเหตุ license ESC-50 (CC BY-NC 3.0):** ใช้เพื่อการเรียนการสอนในห้องได้ฟรี
> แต่ **ห้ามนำไปแจกต่อ/ขาย/รวมลงสื่อที่แจกเชิงพาณิชย์** — repo นี้จึง **ไม่ใส่ไฟล์เสียง
> มาให้** ทุกคนต้องดาวน์โหลดเองครั้งเดียว (เซลล์ถัดไปจัดการให้) ถ้าวันหนึ่งต้องการ
> สื่อที่แจกต่อเชิงพาณิชย์ได้ ให้สลับไปใช้ **Speech Commands v2 (CC BY 4.0)**
> แทน — โครงสร้างโค้ดเปลี่ยนน้อยมาก

## 0. โน้ตช่วย — คำศัพท์ที่ใช้ในโมดูลนี้

| คำ | ความหมาย |
|---|---|
| **spectrogram** | เสียงแปลงเป็น "ภาพ": แกนนอน = เวลา, แกนตั้ง = ความถี่, สี/ความสว่าง = ความดัง — CNN อ่านได้เพราะมันคือตารางตัวเลขจริง ๆ |
| **log-mel spectrogram** | spectrogram ที่พับความถี่ด้วยสเกล mel (ตามหูมนุษย์) แล้ววัดด้วยหน่วย dB — ขนาด 128 × 216 ต่อคลิป 5 วินาที |
| **MFCC** (ครึ่งแรก) | การบีบ log-mel spectrogram ด้วย DCT — เหลือ ~20 ตัวเลขต่อเฟรม แล้วสรุปเป็น mean/std 40 ตัวเลข |
| **CNN** | โครงข่ายประสาทเทียมที่ใช้ตัวกรอง (convolution) กวาดหา pattern ในภาพ — เหมือน M1 ที่คุณเทรนจากศูนย์บน Fashion-MNIST |
| **fold** | ESC-50 แบ่ง 5 folds มาให้แล้ว (กัน leakage จากไฟล์ต้นทางเดียวกัน) — ที่นี่ใช้ protocol ย่อ: เทรน folds 1–4, ทดสอบ fold 5 เพื่อให้จบในเวลาห้องเรียน |
| **epoch / batch** | หนึ่งรอบการอ่านข้อมูลเทรนทั้งชุด / จำนวนตัวอย่างที่ป้อนครั้งละก้อน (ที่นี่ 32 คลิป) |

In [ ]:
import os  # noqa: E402

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: ไม่มี GPU — โมดูลนี้ออกแบบให้รันบน Kaggle T4")

## 0.1 ดาวน์โหลด ESC-50 (บน Kaggle — Internet: On)

ดาวน์โหลด zip ~600 MB จาก GitHub แล้วแตกไฟล์ลง `/kaggle/tmp` (พื้นที่ชั่วคราว
ของ Kaggle — ไม่ติดไปกับ output ของ notebook) — รันครั้งแรกใช้เวลา 2–5 นาที

In [ ]:
from pathlib import Path  # noqa: E402

import requests  # noqa: E402
import zipfile  # noqa: E402

ESC50_URL = "https://github.com/karoldvl/ESC-50/archive/master.zip"


def ensure_esc50() -> Path:
    """ดาวน์โหลด/แตกไฟล์ ESC-50 ครั้งเดียว แล้วคืน path ของโฟลเดอร์."""
    on_kaggle = Path("/kaggle").exists()
    base = (
        Path("/kaggle/tmp") if on_kaggle
        else Path.home() / ".cache" / "esc50"
    )
    found = base / "ESC-50-master"
    if (found / "meta" / "esc50.csv").exists():
        print("ใช้ ESC-50 ที่มีอยู่แล้ว:", found)
        return found
    base.mkdir(parents=True, exist_ok=True)
    zpath = base / "esc50-master.zip"
    if not zpath.exists():
        print("ดาวน์โหลด ESC-50 (~600 MB) — ใช้เวลา 2–5 นาที")
        with requests.get(ESC50_URL, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(zpath, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(base)
    print("แตกไฟล์แล้วที่:", found)
    return found


ESC50 = ensure_esc50()


In [ ]:
meta = pd.read_csv(ESC50 / "meta" / "esc50.csv")
id2cat = (
    meta.drop_duplicates("target")
    .sort_values("target")["category"]
    .tolist()
)
paths = [ESC50 / "audio" / f for f in meta["filename"]]
y_all = meta["target"].to_numpy()
folds = meta["fold"].to_numpy()
SR = 22050
print("จำนวนคลิป:", len(meta), "| จำนวน class:",
      meta["category"].nunique())

## 1. เสียง → ภาพ: log-mel spectrogram

ตัวเลขดิบของคลิป 5 วินาทีมี ~110,250 ค่า — จัดเรียงใหม่เป็น spectrogram
แล้วมันคือ **ภาพขาวดำ 128 × 216 พิกเซล** ที่แต่ละ class มี texture จำเพาะ
(เสียงกระแทก = เส้นตั้ง, โทนต่อเนื่อง = แถบนอน, เสียงสัตว์ร้อง = เส้นโค้ง) —
จุดนี้แหละที่ CNN จาก M1 ใช้ได้พอดี

In [ ]:
import librosa  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
from IPython.display import Audio, display  # noqa: E402

SR = 22050
EXAMPLE = "1-100032-A-0.wav"  # เสียงหมาเห่า (class: dog)
y, _ = librosa.load(ESC50 / "audio" / EXAMPLE, sr=SR, mono=True)
print(f"{EXAMPLE} — {len(y) / SR:.1f} วินาที @ {SR} Hz "
      f"= {len(y):,} ตัวเลขดิบต่อคลิป")
display(Audio(data=y, rate=SR))

mel = librosa.power_to_db(
    librosa.feature.melspectrogram(y=y, sr=SR, n_mels=128),
    ref=np.max,
)
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(np.arange(len(y)) / SR, y, lw=0.4)
axes[0].set_title("waveform — คลื่นเสียงดิบ")
axes[0].set_xlabel("เวลา (วินาที)")
img = axes[1].imshow(mel, aspect="auto", origin="lower", cmap="magma")
axes[1].set_title("log-mel spectrogram — มุมมองแบบ 'ภาพ'")
axes[1].set_xlabel("เวลา (หน่วยเฟรม)")
axes[1].set_ylabel("ความถี่ (mel)")
fig.colorbar(img, ax=axes[1], label="dB")
plt.tight_layout()
plt.show()


## 2. แปลงทุกคลิป → ภาพ spectrogram 2,000 ใบ

คำนวณครั้งเดียวเก็บใน RAM (~220 MB) — ทุกคลิปได้ขนาดเท่ากัน (128 × 216)
จึงกองเป็น tensor เดียวได้ แล้วแบ่งตาม fold ของ ESC-50: **เทรน folds 1–4
(1,600 คลิป) ทดสอบ fold 5 (400 คลิป)**

In [ ]:
from joblib import Parallel, delayed  # noqa: E402


import librosa  # noqa: E402


def mel_row(path) -> np.ndarray:
    """คลิปเดียว → log-mel spectrogram (128, 216) float32."""
    y, _ = librosa.load(path, sr=SR, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=128)
    return librosa.power_to_db(S).astype(np.float32)


mels = np.asarray(Parallel(n_jobs=-1)(
    delayed(mel_row)(p) for p in paths
))
print("mels:", mels.shape, f"≈ {mels.nbytes / 1e6:.0f} MB ใน RAM")

tr, te = folds != 5, folds == 5
Xtr = torch.from_numpy(mels[tr]).unsqueeze(1)  # (N, 1, 128, 216)
Xte = torch.from_numpy(mels[te]).unsqueeze(1)
ytr = torch.from_numpy(y_all[tr])
yte = torch.from_numpy(y_all[te])

# normalize ด้วยสถิติของ train เท่านั้น (กัน leakage แบบเดียวกับ M1)
mean, std = Xtr.mean(), Xtr.std()
Xtr_n = (Xtr - mean) / std
Xte_n = (Xte - mean) / std
print(f"normalize: mean={mean:.2f} std={std:.2f}")
print("train:", tuple(Xtr_n.shape), "| test (fold 5):",
      tuple(Xte_n.shape))

## 3. baseline สำหรับเทียบ — RF บน MFCC (ครึ่งแรกของโมดูล)

เทรน RF ตัวเดียวกับโมดูล M4-RF แต่ใช้ split เดียวกับ CNN (เทรน folds 1–4,
ทดสอบ fold 5) เพื่อให้เทียบกัน **บนข้อมูลทดสอบชุดเดียวกันจริง ๆ** — ใช้เวลา
~3–5 นาทีบน Kaggle CPU (4 คอร์)

In [ ]:
from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.metrics import accuracy_score  # noqa: E402


def mfcc_row(path) -> np.ndarray:
    """คลิปเดียว → เวกเตอร์ 40 ตัวเลข (mean+std ของ MFCC 20 ตัว)."""
    y, _ = librosa.load(path, sr=SR, mono=True)
    m = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=20)
    return np.concatenate([m.mean(axis=1), m.std(axis=1)])


X_mfcc = np.asarray(Parallel(n_jobs=-1)(
    delayed(mfcc_row)(p) for p in paths
))
rf = RandomForestClassifier(
    n_estimators=200, n_jobs=-1, random_state=SEED
).fit(X_mfcc[tr], y_all[tr])
acc_rf5 = accuracy_score(y_all[te], rf.predict(X_mfcc[te]))
print(f"RF บน MFCC — accuracy (fold 5) = {acc_rf5:.4f}")

## 4. CNN จากศูนย์ — อ่าน spectrogram เหมือนอ่านภาพ

สถาปัตยกรรมจิ๋วแบบเดียวกับ M1 แต่ลึกขึ้นนิดเดียวเพราะ "ภาพ" ใหญ่กว่า:
`Conv(32) → MaxPool → Conv(64) → MaxPool → Conv(64) → MaxPool →
Flatten → Linear(50)` — ตัวกรองชั้นแรกจะเรียนรู้ pattern เสียงพื้นฐานเอง
(แถบความถี่, จังหวะกระแทก) โดยไม่มีใครออกแบบให้

เทรน **30 epochs** ด้วย Adam บน 1,600 คลิป (ใช้เวลาราว 4–8 นาทีบน T4):

In [ ]:
import torch.nn as nn  # noqa: E402
from torch.utils.data import DataLoader, TensorDataset  # noqa: E402


class SmallCNN(nn.Module):
    """CNN จิ๋ว 3 ชั้น conv — อ่าน spectrogram (1, in_mels, in_frames)."""

    def __init__(self, in_mels=128, in_frames=216, n_classes=50):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(
                64 * (in_mels // 8) * (in_frames // 8), n_classes
            ),
        )

    def forward(self, x):
        return self.net(x)


def train_cnn(Xa, ya, epochs=30, bs=32, seed=0, log_every=5):
    """เทรน CNN ด้วย Adam บน GPU — คืนโมเดลที่เทรนแล้ว."""
    torch.manual_seed(seed)
    model = SmallCNN(
        in_mels=Xa.shape[2], in_frames=Xa.shape[3]
    ).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    g = torch.Generator().manual_seed(seed)
    dl = DataLoader(
        TensorDataset(Xa, ya), batch_size=bs, shuffle=True,
        generator=g,
    )
    for ep in range(epochs):
        model.train()
        run = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            run += loss.item()
        if log_every and (ep + 1) % log_every == 0:
            print(f"  epoch {ep + 1}/{epochs}: "
                  f"loss={run / len(dl):.4f}")
    return model


def cnn_accuracy(model, Xa, ya):
    model.eval()
    with torch.no_grad():
        pred = model(Xa.to(DEVICE)).argmax(1).cpu().numpy()
    return accuracy_score(ya.numpy(), pred)

In [ ]:
cnn = train_cnn(Xtr_n, ytr, epochs=30)
acc_cnn = cnn_accuracy(cnn, Xte_n, yte)
print(f"CNN จากศูนย์ — accuracy (fold 5) = {acc_cnn:.4f}")

**อ่าน loss:** ต้องลดลงราย epoch (บน T4 แต่ละ epoch ~10 วินาที) — ถ้า
loss ลดแล้วคงที่สูงเกินไป ลองเพิ่ม epochs (ข้อ 1 ของ exercise ให้ลองพอดี)

### 4.1 CNN พลาดอะไร — confusion matrix ของ fold 5

In [ ]:
import matplotlib.pyplot as plt  # noqa: E402
from collections import Counter  # noqa: E402
from sklearn.metrics import ConfusionMatrixDisplay  # noqa: E402

with torch.no_grad():
    pred_cnn = cnn(Xte_n.to(DEVICE)).argmax(1).cpu().numpy()

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(
    y_all[te], pred_cnn, ax=ax, colorbar=False,
    display_labels=id2cat, xticks_rotation=90,
)
ax.set_title("confusion matrix — fold 5 (CNN บน log-mel spectrogram)")
plt.tight_layout()
plt.show()

wrong = Counter(
    (id2cat[t], id2cat[p])
    for t, p in zip(y_all[te], pred_cnn) if t != p
)
print("คู่สับสนบ่อยสุด (จริง → ทำนายผิด):")
for (t, p), n in wrong.most_common(8):
    print(f"  {t} → {p}: {n} ครั้ง")

## 5. เทียบสองวิธี — bake-off บนข้อมูลเดียวกัน

ทั้งสองวิธีเพิ่งทำนาย **fold 5 ชุดเดียวกัน** — ตารางนี้เลยเป็นการเทียบที่
ยุติธรรม (ระวัง: CNN ที่นี่ทดสอบ fold เดียว ไม่ใช่ 5-fold mean แบบ paper —
ตัวเลขอาจต่างจากโค้ด reference เล็กน้อย)

In [ ]:
summary = pd.DataFrame({
    "วิธี": [
        "RF บน MFCC (สะพานจาก DS)",
        "CNN จากศูนย์ บน log-mel",
    ],
    "input ต่อคลิป": [
        "ตาราง 40 ตัวเลข (mean+std)",
        "ภาพ 128×216 ค่า dB",
    ],
    "ที่รัน": ["CPU ~5 นาที", "Kaggle T4 ~5–10 นาที"],
    "accuracy (fold 5)": [
        f"{acc_rf5:.4f}",
        f"{acc_cnn:.4f}",
    ],
})
print(summary.to_string(index=False))

### Trade-off ที่ต้องจำ (สรุปของโมดูล)

- **RF + MFCC:** เร็ว, รันบน CPU ธรรมดา, อธิบายได้ (feature importance),
  ใช้ทักษะคอร์ส DS เดิมทั้งหมด — เหมาะเป็น baseline แรกของทุกโจทย์เสียง
  แต่เพดาน accuracy ต่ำกว่าเพราะการบีบ 110,250 ตัวเลข → 40 ตัวเลข ตัด
  texture ละเอียดทิ้ง
- **CNN + spectrogram:** accuracy สูงกว่าชัดเจน (คาดหมายกว้าง ~0.60–0.75 —
  จดค่าที่คุณรันได้จริงลงตารางด้านบน) เพราะเรียนรู้ feature เองจาก spectrogram
  เต็ม — แต่ต้องมี GPU, เทรนช้ากว่า 10 เท่า, อธิบายยากกว่า
- ตัวเลขอ้างอิงจาก paper: RF baseline (MFCC & ZCR) = **44.3%** · CNN baseline
  ของ Piczak = **64.5%** · คนจริง = **81.3%** — CNN ยุคใหม่ที่ pretrained มาแล้ว
  แตะ 95%+ แต่ไม่ใช่โจทย์ของโมดูล "จากศูนย์" นี้
- สำหรับงานเมโทรโลยี: เริ่ม RF ก่อนเสมอ แล้วถามว่า accuracy ที่เพิ่มได้
  คุ้มต้นทุน GPU/ความซับซ้อน/ความอธิบายได้ไหม

**และอย่าลืม:** ข้อมูล ESC-50 เป็น **CC BY-NC 3.0** — ห้ามนำผลงานที่เทรนจาก
ชุดนี้ไปแจกต่อเชิงพาณิชย์ ถ้าโจทย์จริงต้องแจกต่อ ให้สลับไป Speech Commands v2
(CC BY 4.0)

## สรุป

ครึ่งแรกของโมดูล (RF) พิสูจน์ว่า "เสียง → ตาราง 40 ตัวเลข → RF" ได้ ~0.45 —
พอปลดข้อจำกัดการบีบ feature ด้วยมือ ให้ CNN อ่าน spectrogram เต็ม จำแนก 50
class เสียงสิ่งแวดล้อมได้ดีขึ้นชัดเจนด้วยโครงข่ายจิ๋วที่คุณเขียนเองได้ใน ~40 บรรทัด
— กฎเดียวกับ M1: **ยิ่งให้โมเดลเรียนรู้ feature เอง ยิ่งต้องแลกด้วยข้อมูล
และพลังประมวลผล**

**ย้อนดู:** `day2/module4_sound_to_numbers/book.ipynb` (RF, offline CPU) ·
ลองเองใน `exercise.ipynb` · เฉลยใน `solution.ipynb`